# 7. Inferencia 

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import os
from Hicherical_TTC import HierarchicalClassifier

## Configuración

In [2]:
# ============================================
# CONFIGURACIÓN - MODIFICAR AQUÍ
# ============================================

CHECKPOINT_PATH = './outputs_fede/best_model_final.pt'
TOKENIZER_PATH = './dataset_preparado_fede/tokenizer' 
INPUT_PICKLE = 'df_garticul.pkl' 
TEXT_COLUMN = 'texto'
OUTPUT_FILE = 'predicciones_fede.pkl'
BATCH_SIZE = 32
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

Device: cuda


## Cargar Checkpoint y Configuración

In [3]:
print(f"Carregant checkpoint: {CHECKPOINT_PATH}")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
config = checkpoint['config']
level_trained = checkpoint.get('level_trained', 5)

print(f"\nConfiguració detectada:")
print(f"   Model: {config['model_name']}")
print(f"   Nivell entrenat: {level_trained}")
print(f"   Max length: {config['max_length']}")

Carregant checkpoint: ./outputs_fede/best_model_final.pt

Configuració detectada:
   Model: Alibaba-NLP/gte-multilingual-base
   Nivell entrenat: 5
   Max length: 512


In [4]:
# Cargar tokenizer
if os.path.exists(TOKENIZER_PATH):
    print(f"Carregant tokenizer desde: {TOKENIZER_PATH}")
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH)
else:
    print(f"Carregant tokenizer desde: {config['model_name']}")
    tokenizer = AutoTokenizer.from_pretrained(config['model_name'], token=HF_TOKEN)

Carregant tokenizer desde: ./dataset_preparado_fede/tokenizer


## Crear Model i Carregar Pesos

In [5]:
print(f"Creant model Level {level_trained}...")

# Detectar attention_mode del checkpoint
attention_mode = checkpoint.get('attention_mode', 'hard_mask')
print(f"   Attention mode: {attention_mode}")

model = HierarchicalClassifier(
        model_name=config['model_name'],
        num_classes=checkpoint['num_classes'],
        hierarchy_mappings=checkpoint['hierarchy_mappings'],
        num_heads=config['num_heads'],
        dropout=config['dropout'],
        attention_mode='ttc_pure', 
    )

print("Carregant pesos del checkpoint...")

# Carregar state_dict amb compatibilitat per noms antics de buffers
state_dict = checkpoint['model_state_dict']

# Convertir noms de mask_L* a transition_L* si cal
new_state_dict = {}
for key, value in state_dict.items():
    if key.startswith('mask_L') and '_to_L' in key:
        # Convertir mask_L1_to_L2 -> transition_L1_to_L2
        new_key = key.replace('mask_L', 'transition_L')
        new_state_dict[new_key] = value
        print(f"   Convertit: {key} -> {new_key}")
    else:
        new_state_dict[key] = value

model.load_state_dict(new_state_dict, strict=False)
model = model.to(device)
model.eval()

print(f"✓ Modelo carregado i listo para inferencia!")
print(f"   Mode: {model.attention_mode}")

if device == 'cuda':
    print(f"   Memoria GPU: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


Creant model Level 5...
   Attention mode: hard_mask
Construint Classificador Jerarquic ...
   Encoder: Alibaba-NLP/gte-multilingual-base
   Nivells: 4
   Classes per nivell: [3, 10, 49, 167]
   Mode d'atencio: ttc_pure
   Heads: 12
   CLS tokens: 4 (posicions 0-3)
   Query size: 167 per nivell


2026-02-28 09:13:24.521753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772270004.537283    1771 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772270004.542235    1771 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-28 09:13:24.557845: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Some weights of the model checkpoint at Alibaba-NLP/gte-multilingual-base were not used when initializing N

   Total parametres: 334,584,805
Carregant pesos del checkpoint...
✓ Modelo carregado i listo para inferencia!
   Mode: ttc_pure
   Memoria GPU: 2.52 GB


## Dataset per Inferència

In [10]:
class InferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, text_column):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.text_column = text_column
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row[self.text_column])
        encoding = self.tokenizer(
            text, max_length=self.max_length, padding='max_length',
            truncation=True, return_tensors='pt', add_special_tokens=False
        )
        return {
            'codigo':row['codigo'],
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'idx': idx
        }

## Carregar Dades d'Entrada

In [11]:
print(f"Carregant dades: {INPUT_PICKLE}")
df = pd.read_pickle(INPUT_PICKLE)
print(f"Vocabulario: {len(tokenizer)} tokens")
print(tokenizer.special_tokens_map)
cls_token = tokenizer.cls_token
sep_token = tokenizer.sep_token
df['texto'] = cls_token+' '+cls_token+' '+cls_token+' '+cls_token+' '+df['texto'].str.replace('sep_token',sep_token)
print(f"   Mostres: {len(df)}")
print(f"   Columnes: {list(df.columns)}")
df.head()

Carregant dades: df_garticul.pkl
Vocabulario: 250002 tokens
{'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}
   Mostres: 45150
   Columnes: ['codigo', 'nomart', 'level1', 'level2', 'level3', 'level4', 'texto']


,codigo,nomart,level1,level2,level3,level4,texto
0,00004984,MACOESA ANISHINO INF 25 BOLSAS,82,89,891,8916,<s> <s> <s> <s> MACOESA ANISHINO INF 25 BOLSAS...
1,00008286,APOSITO PRIMAPORE ADH 15X8 CM 20 UN,86,86,864,8643,<s> <s> <s> <s> APOSITO PRIMAPORE ADH 15X8 CM ...
2,00008491,CLENOSAN LECHE CORPORAL 200 ML,81,81,814,8146,<s> <s> <s> <s> CLENOSAN LECHE CORPORAL 200 ML...
3,00009682,JOBST MEDIA COMPRINET-PRO T/2 LAR BLN 2 UN 463...,86,84,842,8420,<s> <s> <s> <s> JOBST MEDIA COMPRINET-PRO T/2 ...
4,00012179,SOLGAR GINKGO BILOBA 90 MG 60 VCAP,82,89,897,8973,<s> <s> <s> <s> SOLGAR GINKGO BILOBA 90 MG 60 ...


In [12]:
dataset = InferenceDataset(df, tokenizer, config['max_length'], TEXT_COLUMN)

In [14]:
example = dataset.__getitem__(0)
example

{'codigo': '00004984',
 'input_ids': tensor([     0,      0,      0,      0,  78826,    670,  54761,  14368, 186631,
           8575,   5881,    919,    714,      6,  98335,  93818,      2,    663,
           1679,    220,   3488,      7,    913,      9, 110828,     31,     22,
            220,    308,    596,    158,  20286,  26364,   7772,    246,    113,
          39078,  64722,    147,  18602,  18464,      5, 159544,    116,   3488,
              7,   5232,      9, 110828,     31,    144,   3879,      4,  20034,
              8,    576,  49794,  37128,      7,      5,    436,  87520,  96389,
           1866,  11832,     15,     66,  17816,  17258,     16,  86475,      4,
          21501,   3031, 226887,   4086,     15,  27641,    433,     16,  28602,
              4,    891,    150,  45071,    493,    316,     15,     66,  17816,
         114566,    246,     16,  12719,      4,   9572,   4300,  68510, 129474,
             13,     15,   4505,  29210,     16,  12719,      5,      1, 

In [19]:
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

## Crear DataFrame Resultats

In [22]:
print(f"Inferint {len(df)} mostres...")
all_predictions = []
idx_to_labels = checkpoint['idx_to_labels']
USE_GREEDY = True  # <-- Cambiar a False para usar beam search

print(f"\n{'='*60}")
with torch.no_grad():
    for batch in tqdm(dataloader, desc="Predint"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)        
        results_batch = model(input_ids=input_ids, attention_mask=attention_mask)
        batch_size = input_ids.size(0)
        for i in range(batch_size):
            pred = {'codigo': batch['codigo'][i]}
            pred['pred_level1'] = idx_to_labels[0][results_batch.predictions['L1'][i].item()]
            pred['pred_level2'] = idx_to_labels[1][results_batch.predictions['L2'][i].item()]
            pred['pred_level3'] = idx_to_labels[2][results_batch.predictions['L3'][i].item()]
            pred['pred_level4'] = idx_to_labels[3][results_batch.predictions['L4'][i].item()]
            pred['probs'] = results_batch.prob_path[i].item()
            all_predictions.append(pred)

print(f"\n✓ Prediccions completades: {len(all_predictions)} mostres")

Inferint 45150 mostres...



Predint: 100%|██████████| 1411/1411 [58:35<00:00,  2.49s/it]


✓ Prediccions completades: 45150 mostres


## Guardar Resultats

In [29]:
predictions_df = pd.DataFrame(all_predictions)
result_df = pd.merge(df,predictions_df,on='codigo')
result_df.head()

,codigo,nomart,level1,level2,level3,level4,texto,pred_level1,pred_level2,pred_level3,pred_level4,probs
0,00004984,MACOESA ANISHINO INF 25 BOLSAS,82,89,891,8916,<s> <s> <s> <s> MACOESA ANISHINO INF 25 BOLSAS...,82,89,891,8916,0.994081
1,00008286,APOSITO PRIMAPORE ADH 15X8 CM 20 UN,86,86,864,8643,<s> <s> <s> <s> APOSITO PRIMAPORE ADH 15X8 CM ...,86,86,864,8643,0.997002
2,00008491,CLENOSAN LECHE CORPORAL 200 ML,81,81,814,8146,<s> <s> <s> <s> CLENOSAN LECHE CORPORAL 200 ML...,81,81,814,8146,0.996747
3,00009682,JOBST MEDIA COMPRINET-PRO T/2 LAR BLN 2 UN 463...,86,84,842,8420,<s> <s> <s> <s> JOBST MEDIA COMPRINET-PRO T/2 ...,86,84,842,8420,0.999549
4,00012179,SOLGAR GINKGO BILOBA 90 MG 60 VCAP,82,89,897,8973,<s> <s> <s> <s> SOLGAR GINKGO BILOBA 90 MG 60 ...,82,89,897,8973,0.997208


In [30]:
if OUTPUT_FILE.endswith('.pkl'):
    result_df.to_pickle(OUTPUT_FILE)
else:
    result_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print(f"✓ Resultats guardats a: {OUTPUT_FILE}")

✓ Resultats guardats a: predicciones_fede.pkl
